## Problem Statement

Objective: melakukan sentimen analysis dari review pembeli menjadi sentimen Positif atau Negatif.
Target/label: Positif/Negative
Batasan/asumsi: Hanya ada 2 label sebagai output dari sentimen analysis dari setiap review pembeli.

## 1. Load Library

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

## 2. Load Dataset

In [3]:
df = pd.read_csv("../data/customer_reviews_sentiment.csv")
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Setup Training Data Set

In [4]:
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


## 4 Model Klasik (Scikit-leard)

### 4.1 Feature Vectorization

In [5]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Train the Model

In [6]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

### 4.3 Prediksi menggunakan Test Set

In [8]:
y_pred_classic = clf.predict(X_test_vec)
y_pred_classic

array(['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif',
       'positif', 'positif', 'negatif', 'negatif', 'positif', 'negatif',
       'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif',
       'negatif', 'negatif', 'positif', 'negatif', 'positif', 'positif',
       'positif', 'positif', 'positif', 'positif', 'positif', 'negatif',
       'positif', 'negatif', 'positif', 'negatif', 'negatif', 'negatif',
       'negatif', 'negatif', 'positif', 'positif'], dtype=object)

## 5. Menggunakan Model LLM - API GEMINI

In [46]:
or_api_key=os.environ.get('OPENROUTER_API_KEY')

In [97]:
import requests
import json

model = "google/gemini-3.8-flash"

def classify_sentiment_llm_OR(review_text):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {or_api_key}",
            "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
            "X-OpenRouter-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
            },
        data=json.dumps({
            "model": f"{model}",
            "messages": [
                {
                  "role": "user",
                  "content": f"{prompt}"
                }]
        })
    )
    return response.text.strip().lower()

In [71]:
y_pred_llm = []
for review in X_test:
    pred = classify_sentiment_llm_OR(review)
    y_pred_llm.append(pred)

In [72]:
y_pred_llm

['{"id":"gen-1790397426-l8eirkiuh7vjkegl7s4h","object":"chat.completion","created":1790397426,"model":"google/gemini-3.8-flash","provider":"google","system_fingerprint":null,"service_tier":"provisioned","choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"negatif","refusal":null,"reasoning":"**classifying sentiment**\\n\\nmy analysis of the provided indonesian phrase, \\"barang tidak berfungsi sama sekali begitu diterima,\\" leads me to classify its sentiment as negative. the core of the message clearly indicates a product that was non-functional upon arrival, which is unequivocally a negative experience.\\n\\n","reasoning_details":[{"type":"reasoning.text","text":"**classifying sentiment**\\n\\nmy analysis of the provided indonesian phrase, \\"barang tidak berfungsi sama sekali begitu diterima,\\" leads me to classify its sentiment as negative. the core of the message clearly indicates a product that was no

### Sentimen Extraction from API Responses

In [74]:
y_pred_llm_clean = []
for pred in y_pred_llm:
    obj = json.loads(pred)
    content = obj["choices"][0]["message"]["content"]
    y_pred_llm_clean.append(content)
y_pred_llm_clean

['negatif',
 'positif',
 'positif',
 'positif',
 'negatif',
 'negatif',
 'positif',
 'positif',
 'negatif',
 'negatif',
 'positif',
 'negatif',
 'positif',
 'positif',
 'negatif',
 'positif',
 'negatif',
 'positif',
 'negatif',
 'negatif',
 'positif',
 'negatif',
 'positif',
 'positif',
 'positif',
 'positif',
 'positif',
 'positif',
 'positif',
 'negatif',
 'positif',
 'negatif',
 'positif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'negatif',
 'positif',
 'positif']

## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Klasik

In [68]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.2 Evaluasi LLM

In [75]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm_clean))
print(classification_report(y_test, y_pred_llm_clean))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm_clean))

=== LLM API (Gemini) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.3 Tabel Perbandingan

In [103]:
df_accuracy = pd.DataFrame([accuracy_score(y_test, y_pred_classic), accuracy_score(y_test, y_pred_llm_clean)], index=['Model Klasik', f'LLM API {model}'], columns=['Accuracy'])
df_classification_report_classic=pd.DataFrame(classification_report(y_test, y_pred_classic, output_dict=True)).T
df_classification_report_llm = pd.DataFrame(classification_report(y_test, y_pred_llm_clean, output_dict=True)).T
df_cm_classic= pd.DataFrame(confusion_matrix(y_test, y_pred_classic), index=['Negatif', 'Positif'], columns=['Negatif', 'Positif'])
df_cm_llm= pd.DataFrame(confusion_matrix(y_test, y_pred_llm_clean), index=['Negatif', 'Positif'], columns=['Negatif', 'Positif'])
display(df_accuracy)
print('==CLASSIFICATION REPORT==')
print('=== Model Klasik (Scikit-learn) ===')
display(df_classification_report_classic)
print('=== Confusion Matrix Model Klasik (Scikit-learn) ===')
display(df_cm_classic)
print(f'=== LLM API ({model}) ===')
display(df_classification_report_llm)
print(f'=== Confusion Matrix LLM API ({model}) ===')
display(df_cm_llm)

,Accuracy
Model Klasik,1.0
LLM API google/gemini-3.8-flash,1.0


==CLASSIFICATION REPORT==
=== Model Klasik (Scikit-learn) ===


,precision,recall,f1-score,support
negatif,1.0,1.0,1.0,18.0
positif,1.0,1.0,1.0,22.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,40.0
weighted avg,1.0,1.0,1.0,40.0


=== Confusion Matrix Model Klasik (Scikit-learn) ===


,Negatif,Positif
Negatif,18,0
Positif,0,22


=== LLM API (google/gemini-3.8-flash) ===


,precision,recall,f1-score,support
negatif,1.0,1.0,1.0,18.0
positif,1.0,1.0,1.0,22.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,40.0
weighted avg,1.0,1.0,1.0,40.0


=== Confusion Matrix LLM API (google/gemini-3.8-flash) ===


,Negatif,Positif
Negatif,18,0
Positif,0,22


## 7. Analisis Trade Off dan Limitation

Menggunakan LLM dalam melakukan klasifikasi bisa dikatakan "Overshoot" karena menggunakan resource yang berlebihan, dilihat dari:
1. Kebutuhan API ke model LLM, perlu autentikasi dan coding API.
2. Membutuhkan Token.
3. Membutuhkan Prompting yang mengarahkan untuk membuat response hanya Positif atau Negatif.
4. Membutuhkan waktu pemerosesan lebih lama (2 Menitan).

## 8. Rekomendasi Technical Approach

Rekomendasi akhir untuk klasifikasi sentimen bisa menggunakan Metode Klasik dengan library sklearn yang lebih simple, dengan tingkat akurasi yang sama jika menggunakan API LLM (Resource lebih tinggi).